# Aether Stage 2 — clean-audio inference

Four-cell inference notebook for an RTX 3060 or larger NVIDIA GPU.

1. Clone the `stage2` branch and install dependencies.
2. Authenticate and download every required checkpoint.
3. Construct Mimi + AetherSpeech + Connector + 4-bit Qwen3-4B.
4. Set `AUDIO_PATH` and run the complete pipeline, producing both a diagnostic transcript and an experimental direct answer.

The audio should contain one clean spoken question. WAV, FLAC and other formats supported by `soundfile` are accepted.

In [ ]:
# CELL 1 — clone Stage 2 and install every dependency
import logging
import os
import subprocess
import sys
from pathlib import Path

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    force=True,
)

REPO_URL = "https://github.com/karl4th/aether-v3.git"
if Path("/content").exists():
    REPO_DIR = Path("/content/aether-v3")
else:
    REPO_DIR = Path.home() / "aether-v3-runtime"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "stage2"], check=True)
else:
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            "stage2",
            "--single-branch",
            REPO_URL,
            str(REPO_DIR),
        ],
        check=True,
    )

subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "stage2"], check=True)
subprocess.run(
    ["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/stage2"],
    check=True,
)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / "src"))

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-e",
        str(REPO_DIR),
        "bitsandbytes",
        "accelerate",
        "huggingface_hub",
    ],
    check=True,
)

for module_name in list(sys.modules):
    if module_name == "aether_v3" or module_name.startswith("aether_v3."):
        del sys.modules[module_name]

GIT_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    text=True,
).strip()
print("CELL 1 COMPLETE:", {"repo": str(REPO_DIR), "commit": GIT_COMMIT}, flush=True)

In [ ]:
# CELL 2 — authenticate and download all checkpoints
import getpass

from huggingface_hub import hf_hub_download

from aether_v3.config import load_config

try:
    from google.colab import userdata
except ImportError:
    userdata = None

if "HF_TOKEN" not in os.environ:
    colab_token = userdata.get("HF_TOKEN") if userdata is not None else None
    os.environ["HF_TOKEN"] = colab_token or getpass.getpass("Hugging Face read token: ")
assert os.environ["HF_TOKEN"], "A Hugging Face read token is required"

CONFIG_PATH = REPO_DIR / "configs/stage2_r1_full.yaml"
cfg = load_config(CONFIG_PATH)

STAGE1_REPO = "manifestro/aetherASR-EN-v0.1"
STAGE1_FILE = "last.pt"
STAGE2_REPO = "manifestro/aether"
STAGE2_FILE = "stage2/best_wer.pt"

stage1_path = hf_hub_download(
    repo_id=STAGE1_REPO,
    filename=STAGE1_FILE,
    revision=cfg.stage2_train.stage1_revision,
    token=os.environ["HF_TOKEN"],
)
stage2_path = hf_hub_download(
    repo_id=STAGE2_REPO,
    filename=STAGE2_FILE,
    token=os.environ["HF_TOKEN"],
)

assert Path(stage1_path).is_file()
assert Path(stage2_path).is_file()
print(
    "CELL 2 COMPLETE:",
    {"stage1": stage1_path, "stage2": stage2_path, "qwen": cfg.llm.model_id},
    flush=True,
)

In [ ]:
# CELL 3 — construct Mimi + AetherSpeech + Connector + 4-bit Qwen
import dataclasses
import gc
import time

import numpy as np
import soundfile as sf
import torch
import torchaudio
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from aether_v3.models.aether_speech_llm import AetherSpeechLLM
from aether_v3.models.mimi_wrapper import FrozenMimi
from aether_v3.training.stage2_utils import load_stage1_encoder
from aether_v3.training.train_stage2 import load_stage2_trainable_weights

assert torch.cuda.is_available(), "An NVIDIA CUDA GPU is required"
device = torch.device("cuda")

quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    cfg.llm.model_id,
    revision=cfg.llm.revision,
    token=os.environ["HF_TOKEN"],
)
print("QWEN: loading 4-bit weights", flush=True)
llm = AutoModelForCausalLM.from_pretrained(
    cfg.llm.model_id,
    revision=cfg.llm.revision,
    token=os.environ["HF_TOKEN"],
    quantization_config=quantization,
    device_map={"": 0},
    dtype=torch.float16,
    attn_implementation="sdpa",
)

inference_llm_cfg = dataclasses.replace(
    cfg.llm,
    dtype="float16",
    gradient_checkpointing=False,
)
model = AetherSpeechLLM(
    cfg.aether_speech,
    cfg.connector,
    inference_llm_cfg,
    speech_frozen=True,
    llm=llm,
)
stage1_checkpoint = load_stage1_encoder(stage1_path, model.encoder)
model.encoder.to(device)
model.connector.to(device=device, dtype=torch.float16)
stage2_checkpoint = load_stage2_trainable_weights(model, stage2_path, device)
model.eval()

# Mimi starts on CPU and moves to CUDA only while encoding the selected file.
mimi = FrozenMimi(
    cfg.mimi.pretrained_id,
    cfg.mimi.num_quantizers,
    device="cpu",
)

assert model.connector.bridge.output_scale.dtype == torch.float32
print(
    "CELL 3 COMPLETE:",
    {
        "gpu": torch.cuda.get_device_name(0),
        "vram_gb": torch.cuda.get_device_properties(0).total_memory / 2**30,
        "stage1_step": stage1_checkpoint.get("step"),
        "stage2_step": stage2_checkpoint.get("step"),
        "output_scale": float(model.connector.bridge.output_scale.detach()),
        "allocated_gb": torch.cuda.memory_allocated() / 2**30,
    },
    flush=True,
)

In [ ]:
# CELL 4 — set the file and run audio -> Mimi -> AetherSpeech -> Connector -> Qwen
AUDIO_PATH = Path("/content/question.wav")  # <-- change only this path
MAX_TRANSCRIPT_TOKENS = 256
MAX_ANSWER_TOKENS = 96

assert AUDIO_PATH.is_file(), f"Audio file not found: {AUDIO_PATH}"

waveform, sample_rate = sf.read(AUDIO_PATH, dtype="float32", always_2d=True)
waveform = waveform.mean(axis=1)
assert waveform.size > 0, "Audio is empty"
assert np.isfinite(waveform).all(), "Audio contains NaN or Inf"

if sample_rate != cfg.mimi.sampling_rate:
    waveform = torchaudio.functional.resample(
        torch.from_numpy(waveform),
        sample_rate,
        cfg.mimi.sampling_rate,
    ).numpy()
    sample_rate = cfg.mimi.sampling_rate

duration = len(waveform) / sample_rate
peak = float(np.abs(waveform).max(initial=0.0))
assert duration >= 0.25, f"Audio is too short: {duration:.2f}s"
assert peak > 1e-4, f"Audio appears silent: peak={peak}"

pipeline_started = time.time()
torch.cuda.reset_peak_memory_stats()

print("MIMI: encoding", flush=True)
mimi.model.to(device)
mimi.device = device
semantic_codes = mimi.encode_semantic([waveform])[0]
mimi.model.to("cpu")
mimi.device = torch.device("cpu")
gc.collect()
torch.cuda.empty_cache()

codes = torch.from_numpy(semantic_codes).to(device=device, dtype=torch.long).unsqueeze(0)
speech_mask = torch.ones_like(codes, dtype=torch.bool)
with torch.inference_mode():
    speech_states = model.encoder(codes, speech_mask)

assert speech_states.shape == (1, len(semantic_codes), cfg.aether_speech.hidden_size)

def generate(prompt, max_new_tokens):
    prefix = tokenizer(prompt, add_special_tokens=False, return_tensors="pt")
    batch = {
        "speech_states": speech_states,
        "speech_mask": speech_mask,
        "prefix_ids": prefix["input_ids"].to(device),
        "prefix_mask": prefix["attention_mask"].to(device=device, dtype=torch.bool),
    }
    started = time.time()
    token_ids = model.generate_cached(
        batch,
        eos_token_id=tokenizer.eos_token_id,
        max_new_tokens=max_new_tokens,
        use_kv_cache=True,
    )[0]
    text = tokenizer.decode(token_ids, skip_special_tokens=True).strip()
    return text, time.time() - started

transcript, transcript_seconds = generate(
    "Transcribe the following speech exactly. Output only the transcript:\n",
    MAX_TRANSCRIPT_TOKENS,
)
answer, answer_seconds = generate(
    "Answer the spoken question directly. Give only a short answer:\n",
    MAX_ANSWER_TOKENS,
)

result = {
    "audio": str(AUDIO_PATH),
    "audio_seconds": duration,
    "sample_rate": sample_rate,
    "peak": peak,
    "semantic_frames": len(semantic_codes),
    "transcript": transcript,
    "direct_answer_experimental": answer,
    "transcript_seconds": transcript_seconds,
    "answer_seconds": answer_seconds,
    "pipeline_seconds": time.time() - pipeline_started,
    "peak_allocated_gb": torch.cuda.max_memory_allocated() / 2**30,
}

print("\nTRANSCRIPT:\n", transcript, sep="", flush=True)
print("\nDIRECT ANSWER:\n", answer, sep="", flush=True)
print("\nINFERENCE REPORT:\n", json.dumps(result, indent=2), sep="", flush=True)

result_path = AUDIO_PATH.with_suffix(".aether.json")
result_path.write_text(json.dumps(result, indent=2, ensure_ascii=False))
print("SAVED:", result_path, flush=True)